# 2019-2026 Precipitation: Rain Gauges, Hydrograph, and Thiessen Map

This notebook reads all rain gauge from 2019-2026 taken from Toronto's OpenData, plots the selected gauge's hourly precipitation and cumulative hydrograph, and displays a labelled, interactive Thiessen map of daily rainfall totals. Use the **Day** slider to view each day's totals by gauge.

No watershed boundary was provided, so Thiessen polygons are clipped to the convex hull of the gauge network. The method assigns each area the observation from its nearest gauge; it does not create a smooth rainfall surface or calculate an area-weighted catchment average. The rainfall will be measured in mm.

In [1]:
from pathlib import Path
from html import escape

import folium
import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from branca.colormap import linear
from folium.features import DivIcon, GeoJsonTooltip
from IPython.display import display, HTML
from scipy.spatial import Voronoi
from shapely.geometry import Polygon
import os

%matplotlib inline

# Change these settings to select a gauge or focus the hydrograph on an event.
selected_gauge = "RG_056"
event_start = None  # Example: "2026-10-19 00:00"
event_end = None    # Example: "2026-10-20 00:00"

## Load and validate the rain gauge data
* Loads values from CSVs
* Removes missing values, duplicate rows, duplicate dates, and duplicate rain gauge observations

In [2]:
precipitation_2026 = pd.read_csv("precipitation-data-2026.csv")
precipitation_2026["date"] = pd.to_datetime(precipitation_2026["date"])
precipitation_2026["rainfall"] = pd.to_numeric(precipitation_2026["rainfall"])

In [3]:
folder_path=os.listdir()

# Filter the list to only keep files that end with '.csv'
csv_files = [file for file in folder_path if file.endswith('.csv')]

# Concatenate all annual rain data into one dataframe.
precipitation = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

# Convert date column to date time format.
precipitation["date"] = pd.to_datetime(precipitation["date"])
precipitation["rainfall"]=pd.to_numeric(precipitation["rainfall"])

# Remove rows with missing or invalid values in essential columns
precipitation = precipitation.dropna(
    subset=["date", "name", "longitude", "latitude", "rainfall"]
).copy()

# Remove rows with invalid rainfall values
precipitation = precipitation[precipitation["rainfall"] >= 0]

# Remove exact duplicate rows
precipitation = precipitation.drop_duplicates()

# Remove duplicate date-gauge observations
# Keep the first observation for each date and gauge
precipitation = precipitation.drop_duplicates(
    subset=["date", "name"],
    keep="first"
)

# Creating a dataframe of unique rain gauges. Drop duplicates to get unique gauges, sort by name in ascending order, and set the index as the name.
gauges = (
    precipitation[["name", "longitude", "latitude"]]
    .drop_duplicates(subset="name")
    .sort_values("name")
    .reset_index(drop=True)
)

# Indexes the dataframe by the date.
rainfall_by_time = precipitation.pivot(index="date", columns="name", values="rainfall").reindex(columns=gauges["name"])

print(f"Available period: {rainfall_by_time.index.min()} to {rainfall_by_time.index.max()}")
print(f"Total Rainfall observations: {rainfall_by_time.size}")
print(f"0 mm Rainfall observations: {rainfall_by_time.isna().sum().sum():,}")


MemoryError: Unable to allocate 1.23 GiB for an array with shape (3, 54825222) and data type float64

In [ ]:
precipitation.head()

## Splitting Dataframes (3 Season Rain Gauges and 4 Season Rain Gauges)

The city of Toronto has two sets of rain gauges:
1- 3 season rain gauges: they operate between Apr 1st to Oct 31st - We will assume all precipitation captured by them is rain
2- 4 season rain guages: they operate year round. They capture winter precipitation which we will assume is all snow

From city of Toronto Documentation


The tentative list of 3-season gauges is as follows: 
RG_001, RG_002, RG_003, RG_004, RG_006, RG_007, RG_012, RG_013, RG_014, RG_015, RG_020, RG_021
RG_022, RG_023, RG_024, RG_025, RG_027, RG_028, RG_030, RG_031, RG_033, RG_034, RG_035, RG_036, RG_037, RG_039, RG_040, RG_041, RG_042, RG_044, RG_045, RG_046, RG_047, RG_048, RG_049, RG_050, RG_051, RG_052, RG_053

The tentative list of 4-season gauges is as follows:
 RG_016, RG_017, RG_018, RG_019, RG_038


In [ ]:
gauges

In [ ]:
rain_gauge_gpd=gpd.GeoDataFrame(gauges, geometry=gpd.points_from_xy(gauges.longitude, gauges.latitude), crs="32617").drop(columns=["longitude","latitude"])

In [ ]:
# Load the Toronto neighbourhood boundaries using GeoPandas.
neighbourhoods = gpd.read_file("toronto_neighbourhoods.shp")

# Keep only the neighbourhood-name and geometry columns.
neighbourhoods = neighbourhoods[["FIELD_8", "geometry"]]

# Rename "FIELD_8" to "name".
neighbourhoods.columns = ["name", "geometry"]

# Remove the neighbourhood ID in parentheses from the end of each name.
# Hint: str.replace() with a regular expression may be useful.
neighbourhoods["name"] = neighbourhoods["name"].str.split("(").str[0].str.strip()
#splits the string after "(" and then takes the first index .str[0] then strips the trailing spaces

# Preview the cleaned GeoDataFrame
neighbourhoods.head()

In [ ]:
# Create one figure and axes for all spatial layers.
fig, ax = plt.subplots(figsize=(10, 8))

# Plot neighbourhood boundaries.
# Hint: use facecolor="none" so the other layers remain visible.
neighbourhoods.plot(ax=ax, linewidth=1, facecolor="none")

# Plot rain guages on the same axes.
rain_gauge_gpd.plot(ax=ax, linewidth=2, label="Rain Gauges")

# Add the required labels, title, and legend.
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.set_title("Rain Gauges in the City of Toronto")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
#4 Season Rain Guages
four_season_gauges=gauges[gauges["name"].isin(["RG_016", "RG_017", "RG_018", "RG_019", "RG_038"])].reset_index(drop=True)
four_season_gauges

In [ ]:
#3 season Rain Gauges
three_season_gauges = gauges[~gauges['name'].isin(four_season_gauges['name'])].reset_index(drop=True)
three_season_gauges

In [ ]:
print(precipitation.info())
precipitation

## Analyzing Data Patterns
Using aggregation to find the following information:
* Which rain gauges received the most rainfall during the 2019-2026 period? This could give an idea of which areas receive the most rain overall.
* Biggest rainfall within a day collected. Can potentially show which areas are most susceptible to flash floods.


#### Aggregating into Daily Rainfall

In [ ]:
# Aggregate into daily rainfall for each rain gauge. Output the rain gauges that had the largest daily event.
daily_precipitation = precipitation.groupby(["name", "date"], observed=True)["rainfall"].sum(min_count=1).reset_index()
max_daily_precipitation=daily_precipitation.sort_values("rainfall",ascending=False)
max_daily_precipitation.head(20)

In [ ]:
min_daily_precipitation=daily_precipitation.sort_values("rainfall",ascending=True)
min_daily_precipitation

## Splitting into two different dataframes
The city of Toronto indicated that their rain guages are either a 3-season guage or a 4-season guage. 

- Between Apr 1st and October 31st only the three season rain gauges are turned on 
- Between Nov 1st to March 31st all gauges are turned on

--> Split the dataframe into
- Between Apr 1st and Oct 31st --> all rain gauges --> rain 
- Between Nov 1st to Mar 31st --> only 4 season rain gauges --> Snow

In [ ]:
#extract all records between april 1st and october 31st
all_rain_gauges_precipitation=daily_precipitation[(daily_precipitation["date"].dt.month >= 4)|(daily_precipitation["date"].dt.month >= 10)]
all_rain_gauges_precipitation

In [ ]:
rainfall_by_time

In [ ]:
rainfall_by_time_all_gauges = rainfall_by_time[(rainfall_by_time.index.month >= 4) |(rainfall_by_time.index.month >= 10)]
rainfall_by_time_all_gauges

In [ ]:
#extract winter precipitation between Nov 1st and May 31st
winter_precipitation=daily_precipitation[~daily_precipitation['date'].isin(all_rain_gauges_precipitation['date'])]
winter_precipitation

In [ ]:
#winter months
rainfall_by_time[~rainfall_by_time.index.isin(rainfall_by_time_all_gauges.index)]
rainfall_by_time_winter

## Build Thiessen polygons - Apr 1st to Oct 31st - ALL GUAGES

Gauge coordinates are projected to UTM zone 17N before calculating the Voronoi regions. The study area is the gauge network's convex hull because no watershed boundary was supplied. The slider below changes the daily rainfall totals while keeping all gauge labels visible.

In [ ]:
projected_crs = "EPSG:32617"
gauges_gdf = gpd.GeoDataFrame(
    gauges.copy(),
    geometry=gpd.points_from_xy(gauges["longitude"], gauges["latitude"]),
    crs="EPSG:4326",
).to_crs(projected_crs)
coordinates = np.column_stack([gauges_gdf.geometry.x, gauges_gdf.geometry.y])
if len(np.unique(coordinates, axis=0)) != len(coordinates):
    raise ValueError("Two or more gauges have identical coordinates; Thiessen regions cannot be assigned unambiguously.")
if np.linalg.matrix_rank(coordinates - coordinates.mean(axis=0)) < 2:
    raise ValueError("Gauge locations are collinear; a two-dimensional Thiessen map cannot be created.")

voronoi = Voronoi(coordinates)

def finite_voronoi_regions(voronoi_diagram, radius):
    regions = []
    vertices = voronoi_diagram.vertices.tolist()
    center = voronoi_diagram.points.mean(axis=0)
    ridge_map = {}
    for (point_a, point_b), (vertex_a, vertex_b) in zip(voronoi_diagram.ridge_points, voronoi_diagram.ridge_vertices):
        ridge_map.setdefault(point_a, []).append((point_b, vertex_a, vertex_b))
        ridge_map.setdefault(point_b, []).append((point_a, vertex_a, vertex_b))

    for point_index, region_index in enumerate(voronoi_diagram.point_region):
        region = voronoi_diagram.regions[region_index]
        if all(vertex >= 0 for vertex in region):
            regions.append(region)
            continue

        finite_vertices = [vertex for vertex in region if vertex >= 0]
        for other_point, vertex_a, vertex_b in ridge_map[point_index]:
            if vertex_a >= 0 and vertex_b >= 0:
                continue
            finite_vertex = vertex_a if vertex_a >= 0 else vertex_b
            tangent = voronoi_diagram.points[other_point] - voronoi_diagram.points[point_index]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = voronoi_diagram.points[[point_index, other_point]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            vertices.append((voronoi_diagram.vertices[finite_vertex] + direction * radius).tolist())
            finite_vertices.append(len(vertices) - 1)

        region_coordinates = np.asarray([vertices[index] for index in finite_vertices])
        region_center = region_coordinates.mean(axis=0)
        order = np.argsort(np.arctan2(region_coordinates[:, 1] - region_center[1], region_coordinates[:, 0] - region_center[0]))
        regions.append(np.asarray(finite_vertices)[order].tolist())
    return regions, np.asarray(vertices)

radius = np.ptp(coordinates, axis=0).max() * 4
regions, voronoi_vertices = finite_voronoi_regions(voronoi, radius)
study_area = gauges_gdf.geometry.union_all().convex_hull
polygons = [Polygon(voronoi_vertices[region]).intersection(study_area) for region in regions]
if any(polygon.is_empty or not polygon.is_valid for polygon in polygons):
    raise ValueError("A Thiessen polygon is empty or invalid after clipping to the gauge convex hull.")

thiessen = gpd.GeoDataFrame(
    {"gauge": gauges_gdf["name"].to_list()},
    geometry=polygons,
    crs=projected_crs,
)
thiessen_map = thiessen.to_crs("EPSG:4326")
daily_rainfall_by_date_all_gauges = rainfall_by_time_all_gauges.resample("D").sum(min_count=1)
maximum_daily_rainfall = daily_rainfall_by_date_all_gauges.max().max()
color_max = float(maximum_daily_rainfall) if pd.notna(maximum_daily_rainfall) and maximum_daily_rainfall > 0 else 1.0
color_scale = linear.Blues_09.scale(0, color_max)
color_scale.caption = "Daily rainfall total (mm; scale fixed across dates)"

observed_gauges_per_date = daily_rainfall_by_date_all_gauges.notna().sum(axis=1)
if observed_gauges_per_date.empty or observed_gauges_per_date.max() == 0:
    raise ValueError("No daily rainfall observations are available for the Thiessen map.")
maximum_coverage = observed_gauges_per_date.max()
default_date = observed_gauges_per_date[observed_gauges_per_date.eq(maximum_coverage)].index.max()
default_slider_index = daily_rainfall_by_date_all_gauges.index.get_loc(default_date)
print(f"Thiessen polygons: {len(thiessen)}; default map date: {default_date.date()}; gauges with daily totals: {maximum_coverage}")

## Interactive labelled Thiessen rainfall map - Apr 1st to Oct 31st

Move the slider to view each calendar day's summed rainfall for every gauge. Daily totals are the sum of recorded readings for that day; days with missing readings may be partial. The color scale is fixed across dates for comparison. Hover over a polygon or gauge marker for its name and daily rainfall total; grey polygons have no readings that day.

In [ ]:
def make_thiessen_map(date):
    values = daily_rainfall_by_date.loc[date]
    mapped_polygons = thiessen_map.copy()
    mapped_polygons["rainfall"] = mapped_polygons["gauge"].map(values)

    map_object = folium.Map(tiles="OpenStreetMap", control_scale=True)
    folium.GeoJson(
        mapped_polygons.to_json(),
        name="Thiessen rainfall",
        style_function=lambda feature: {
            "fillColor": color_scale(feature["properties"]["rainfall"])
            if feature["properties"]["rainfall"] is not None else "#bdbdbd",
            "color": "#444444",
            "weight": 1,
            "fillOpacity": 0.55,
        },
        tooltip=GeoJsonTooltip(
            fields=["gauge", "rainfall"],
            aliases=["Gauge", "Daily rainfall total (mm)"],
            localize=True,
        ),
    ).add_to(map_object)

    for row in gauges.itertuples(index=False):
        rainfall = values.get(row.name, np.nan)
        rainfall_text = f"{rainfall:.3f} mm for the day" if pd.notna(rainfall) else "No observations that day"
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=3,
            color="#8b0000",
            fill=True,
            fill_opacity=1,
            tooltip=f"{row.name}: {rainfall_text}",
        ).add_to(map_object)
        folium.Marker(
            location=[row.latitude, row.longitude],
            icon=DivIcon(
                icon_size=(70, 14),
                icon_anchor=(0, 7),
                html=f'<div style="font-size:10px;font-weight:bold;color:#222;background:rgba(255,255,255,0.8);white-space:nowrap">{escape(row.name)}</div>',
            ),
        ).add_to(map_object)

    map_object.fit_bounds([
        [gauges["latitude"].min(), gauges["longitude"].min()],
        [gauges["latitude"].max(), gauges["longitude"].max()],
    ])
    color_scale.add_to(map_object)
    folium.LayerControl().add_to(map_object)
    return map_object

time_slider = widgets.IntSlider(
    value=default_slider_index,
    min=0,
    max=len(daily_rainfall_by_date.index) - 1,
    step=1,
    description="Day",
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width="95%"),
)

def update_thiessen_map(time_index):
    date = daily_rainfall_by_date.index[time_index]
    display(HTML(f"<b>Daily Thiessen rainfall totals for {date.date()}</b>"))
    display(make_thiessen_map(date))

map_output = widgets.interactive_output(update_thiessen_map, {"time_index": time_slider})
display(widgets.VBox([time_slider, map_output]))

## Build Thiessen polygons - Nov 1st to March 31st - WINTER

Gauge coordinates are projected to UTM zone 17N before calculating the Voronoi regions. The study area is the gauge network's convex hull because no watershed boundary was supplied. The slider below changes the daily rainfall totals while keeping all gauge labels visible.

In [ ]:
projected_crs = "EPSG:32617"
four_season_gauges_gdf = gpd.GeoDataFrame(
    four_season_gauges.copy(),
    geometry=gpd.points_from_xy(four_season_gauges["longitude"], four_season_gauges["latitude"]),
    crs="EPSG:4326",
).to_crs(projected_crs)
coordinates = np.column_stack([four_season_gauges_gdf.geometry.x, four_season_gauges_gdf.geometry.y])
if len(np.unique(coordinates, axis=0)) != len(coordinates):
    raise ValueError("Two or more gauges have identical coordinates; Thiessen regions cannot be assigned unambiguously.")
if np.linalg.matrix_rank(coordinates - coordinates.mean(axis=0)) < 2:
    raise ValueError("Gauge locations are collinear; a two-dimensional Thiessen map cannot be created.")

voronoi = Voronoi(coordinates)

def finite_voronoi_regions(voronoi_diagram, radius):
    regions = []
    vertices = voronoi_diagram.vertices.tolist()
    center = voronoi_diagram.points.mean(axis=0)
    ridge_map = {}
    for (point_a, point_b), (vertex_a, vertex_b) in zip(voronoi_diagram.ridge_points, voronoi_diagram.ridge_vertices):
        ridge_map.setdefault(point_a, []).append((point_b, vertex_a, vertex_b))
        ridge_map.setdefault(point_b, []).append((point_a, vertex_a, vertex_b))

    for point_index, region_index in enumerate(voronoi_diagram.point_region):
        region = voronoi_diagram.regions[region_index]
        if all(vertex >= 0 for vertex in region):
            regions.append(region)
            continue

        finite_vertices = [vertex for vertex in region if vertex >= 0]
        for other_point, vertex_a, vertex_b in ridge_map[point_index]:
            if vertex_a >= 0 and vertex_b >= 0:
                continue
            finite_vertex = vertex_a if vertex_a >= 0 else vertex_b
            tangent = voronoi_diagram.points[other_point] - voronoi_diagram.points[point_index]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = voronoi_diagram.points[[point_index, other_point]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            vertices.append((voronoi_diagram.vertices[finite_vertex] + direction * radius).tolist())
            finite_vertices.append(len(vertices) - 1)

        region_coordinates = np.asarray([vertices[index] for index in finite_vertices])
        region_center = region_coordinates.mean(axis=0)
        order = np.argsort(np.arctan2(region_coordinates[:, 1] - region_center[1], region_coordinates[:, 0] - region_center[0]))
        regions.append(np.asarray(finite_vertices)[order].tolist())
    return regions, np.asarray(vertices)

radius = np.ptp(coordinates, axis=0).max() * 4
regions, voronoi_vertices = finite_voronoi_regions(voronoi, radius)
study_area = four_season_gauges_gdf.geometry.union_all().convex_hull
polygons = [Polygon(voronoi_vertices[region]).intersection(study_area) for region in regions]
if any(polygon.is_empty or not polygon.is_valid for polygon in polygons):
    raise ValueError("A Thiessen polygon is empty or invalid after clipping to the gauge convex hull.")

thiessen = gpd.GeoDataFrame(
    {"gauge": four_season_gauges_gdf["name"].to_list()},
    geometry=polygons,
    crs=projected_crs,
)
thiessen_map = thiessen.to_crs("EPSG:4326")
daily_rainfall_by_date_winter = rainfall_by_time_winter.resample("D").sum(min_count=1)
maximum_daily_rainfall = daily_rainfall_by_date_winter.max().max()
color_max = float(maximum_daily_rainfall) if pd.notna(maximum_daily_rainfall) and maximum_daily_rainfall > 0 else 1.0
color_scale = linear.Blues_09.scale(0, color_max)
color_scale.caption = "Daily rainfall total (mm; scale fixed across dates)"

observed_gauges_per_date_winter = rainfall_by_time_winter.notna().sum(axis=1)
if observed_gauges_per_date_winter.empty or observed_gauges_per_date_winter.max() == 0:
    raise ValueError("No daily rainfall observations are available for the Thiessen map.")
maximum_coverage = observed_gauges_per_date_winter.max()
default_date = observed_gauges_per_date_winter[observed_gauges_per_date_winter.eq(maximum_coverage)].index.max()
default_slider_index = rainfall_by_time_winter.index.get_loc(default_date)
print(f"Thiessen polygons: {len(thiessen)}; default map date: {default_date.date()}; gauges with daily totals: {maximum_coverage}")

## Interactive labelled Thiessen rainfall map - Nov 1st to March 31st

Move the slider to view each calendar day's summed rainfall for every gauge. Daily totals are the sum of recorded readings for that day; days with missing readings may be partial. The color scale is fixed across dates for comparison. Hover over a polygon or gauge marker for its name and daily rainfall total; grey polygons have no readings that day.

In [ ]:
def make_thiessen_map(date):
    values = daily_rainfall_by_date_winter.loc[date]
    mapped_polygons = thiessen_map.copy()
    mapped_polygons["rainfall"] = mapped_polygons["gauge"].map(values)

    map_object = folium.Map(tiles="OpenStreetMap", control_scale=True)
    folium.GeoJson(
        mapped_polygons.to_json(),
        name="Thiessen rainfall",
        style_function=lambda feature: {
            "fillColor": color_scale(feature["properties"]["rainfall"])
            if feature["properties"]["rainfall"] is not None else "#bdbdbd",
            "color": "#444444",
            "weight": 1,
            "fillOpacity": 0.55,
        },
        tooltip=GeoJsonTooltip(
            fields=["gauge", "rainfall"],
            aliases=["Gauge", "Daily rainfall total (mm)"],
            localize=True,
        ),
    ).add_to(map_object)

    for row in four_season_gauges.itertuples(index=False):
        rainfall = values.get(row.name, np.nan)
        rainfall_text = f"{rainfall:.3f} mm for the day" if pd.notna(rainfall) else "No observations that day"
        folium.CircleMarker(
            location=[row.latitude, row.longitude],
            radius=3,
            color="#8b0000",
            fill=True,
            fill_opacity=1,
            tooltip=f"{row.name}: {rainfall_text}",
        ).add_to(map_object)
        folium.Marker(
            location=[row.latitude, row.longitude],
            icon=DivIcon(
                icon_size=(70, 14),
                icon_anchor=(0, 7),
                html=f'<div style="font-size:10px;font-weight:bold;color:#222;background:rgba(255,255,255,0.8);white-space:nowrap">{escape(row.name)}</div>',
            ),
        ).add_to(map_object)

    map_object.fit_bounds([
        [four_season_gauges["latitude"].min(), four_season_gauges["longitude"].min()],
        [four_season_gauges["latitude"].max(), four_season_gauges["longitude"].max()],
    ])
    color_scale.add_to(map_object)
    folium.LayerControl().add_to(map_object)
    return map_object

time_slider = widgets.IntSlider(
    value=default_slider_index,
    min=0,
    max=len(daily_rainfall_by_date_winter.index) - 1,
    step=1,
    description="Day",
    continuous_update=False,
    readout=False,
    layout=widgets.Layout(width="95%"),
)

def update_thiessen_map(time_index):
    date = daily_rainfall_by_date_winter.index[time_index]
    display(HTML(f"<b>Daily Thiessen rainfall totals for {date.date()}</b>"))
    display(make_thiessen_map(date))

map_output = widgets.interactive_output(update_thiessen_map, {"time_index": time_slider})
display(widgets.VBox([time_slider, map_output]))